# Model Comparison on D2 Features

This notebook compares three model classes on the same fixed feature representation: the D2 feature set (7 core features + `CCC_80` + `CCC_90`). The feature set, population, and split are all frozen from notebook 02 / Experiment D2. Only the model class and its hyperparameters change.

Models compared:
1. Logistic Regression (same configuration as C/D1/D2)
2. Random Forest
3. Gradient Boosting

Model and hyperparameter selection use validation data only. The test set is evaluated once at the end, after every setting is locked.

In [1]:
import pandas as pd
import numpy as np

pumf = pd.read_csv("../Data_Données/pumf_cchs.csv")

print("PUMF shape:", pumf.shape)

PUMF shape: (67079, 255)


In [2]:
model_data = pumf[pumf["CCC_05"].isin([1, 2])].copy()

model_data["target"] = (model_data["CCC_05"] == 1).astype(int)

print("Modelling population:", model_data.shape[0])
print(model_data["target"].value_counts().sort_index())

Modelling population: 66242
target
0    60248
1     5994
Name: count, dtype: int64


## Feature set

`D2_Core_Hypertension_Cholesterol` from notebook 02: the 7 C features plus `CCC_80` plus `CCC_90`. Same as Experiment D2.

In [3]:
FEATURES = [
    "DHHGAGE",
    "DHH_SEX",
    "EDDVH3",
    "BMI_CLASS",
    "INCDGHH",
    "SDCDGIMM",
    "GEOGPRV",
    "CCC_80",
    "CCC_90"
]

print("Number of features:", len(FEATURES))
print(FEATURES)

Number of features: 9
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80', 'CCC_90']


In [4]:
# Same BMI harmonization as C, D1, D2
model_data["BMI_CLASS"] = np.nan

youth_mask = model_data["DHHGAGE"] == 1
adult_mask = model_data["DHHGAGE"].isin([2, 3, 4, 5])

model_data.loc[youth_mask, "BMI_CLASS"] = model_data.loc[youth_mask, "HWTDGWHO"]
model_data.loc[adult_mask, "BMI_CLASS"] = model_data.loc[adult_mask, "HWTDGISW"]

print(model_data["BMI_CLASS"].value_counts(dropna=False).sort_index())

BMI_CLASS
1.0    26053
2.0    37045
6.0       32
9.0     3112
Name: count, dtype: int64


In [5]:
SPECIAL_CODES = {
    "DHHGAGE": [],
    "DHH_SEX": [],
    "EDDVH3": [9],
    "BMI_CLASS": [6, 9],
    "INCDGHH": [9],
    "SDCDGIMM": [9],
    "GEOGPRV": [],
    "CCC_80": [9],
    "CCC_90": [9]
}

def apply_special_codes(df, special_codes):
    result = df.copy()

    for column, codes in special_codes.items():
        if column in result.columns:
            result[column] = result[column].replace(codes, np.nan)

    return result

clean_model_data = apply_special_codes(model_data, SPECIAL_CODES)

print("Missing values after special-code handling:")
print(clean_model_data[FEATURES].isna().sum())

Missing values after special-code handling:
DHHGAGE         0
DHH_SEX         0
EDDVH3       2276
BMI_CLASS    3144
INCDGHH       947
SDCDGIMM      835
GEOGPRV         0
CCC_80        494
CCC_90        155
dtype: int64


## Population and split

Same general population and same stratified train/validation/test split as A, B, C, D1, D2, and F. No changes here — this notebook only varies the model class.

In [6]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_val_idx, test_idx = train_test_split(
    model_data.index,
    test_size=0.20,
    stratify=model_data["target"],
    random_state=RANDOM_STATE
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.20,
    stratify=model_data.loc[train_val_idx, "target"],
    random_state=RANDOM_STATE
)

print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Training: 42394
Validation: 10599
Test: 13249


In [7]:
X_train = clean_model_data.loc[train_idx, FEATURES]
X_val = clean_model_data.loc[val_idx, FEATURES]
X_test = clean_model_data.loc[test_idx, FEATURES]

y_train = model_data.loc[train_idx, "target"]
y_val = model_data.loc[val_idx, "target"]
y_test = model_data.loc[test_idx, "target"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (42394, 9) (42394,)
Validation: (10599, 9) (10599,)
Test: (13249, 9) (13249,)


## Preprocessing

All 9 features are categorical, same as D2. One preprocessor is fitted on the training data only and reused for all three models, so the comparison isolates the effect of the model class.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, FEATURES)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (42394, 34)
Processed validation shape: (10599, 34)
Processed test shape: (13249, 34)


## Threshold sweep on validation

Same threshold-selection approach used throughout the project: sweep thresholds from 0.10 to 0.90 and lock the one with the highest validation F1. This is reused for every model below.

In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

THRESHOLDS = np.arange(0.10, 0.91, 0.01)

def best_threshold(y_true, prob):
    results = []
    for threshold in THRESHOLDS:
        pred = (prob >= threshold).astype(int)
        results.append({
            "threshold": threshold,
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0),
            "accuracy": accuracy_score(y_true, pred)
        })
    results = pd.DataFrame(results)
    best_row = results.loc[results["f1"].idxmax()]
    return float(best_row["threshold"]), float(best_row["f1"])

## 1. Logistic Regression

Same configuration used in C, D1, and D2: `class_weight="balanced"`, `max_iter=1000`, `random_state=42`. No hyperparameter search needed since this configuration is already fixed by the earlier experiments.

In [10]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train_processed, y_train)

log_reg_val_prob = log_reg.predict_proba(X_val_processed)[:, 1]

log_reg_threshold, log_reg_val_f1 = best_threshold(y_val, log_reg_val_prob)

print("Logistic Regression selected threshold:", round(log_reg_threshold, 3))
print("Validation F1:", round(log_reg_val_f1, 4))

Logistic Regression selected threshold: 0.68
Validation F1: 0.3678


## 2. Random Forest

`RandomForestClassifier` does support `class_weight`, so `class_weight="balanced"` is used directly, same idea as Logistic Regression.

A small grid is tried on the validation set: `n_estimators` in {200, 400} and `max_depth` in {None, 10}. This is not an exhaustive search, just enough to see whether tree depth or forest size matters for this feature set. The config with the highest validation F1 (after its own threshold sweep) is kept.

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf_grid = [
    {"n_estimators": 200, "max_depth": None},
    {"n_estimators": 200, "max_depth": 10},
    {"n_estimators": 400, "max_depth": 10}
]

rf_search_results = []
rf_models = {}

for params in rf_grid:
    rf = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_processed, y_train)

    val_prob = rf.predict_proba(X_val_processed)[:, 1]
    threshold, val_f1 = best_threshold(y_val, val_prob)

    key = f"n_estimators={params['n_estimators']}, max_depth={params['max_depth']}"
    rf_models[key] = rf

    rf_search_results.append({
        "config": key,
        "threshold": threshold,
        "validation_f1": val_f1
    })

rf_search_results = pd.DataFrame(rf_search_results)
print(rf_search_results)

                             config  threshold  validation_f1
0  n_estimators=200, max_depth=None       0.72       0.325197
1    n_estimators=200, max_depth=10       0.68       0.371634
2    n_estimators=400, max_depth=10       0.66       0.369369


In [12]:
best_rf_row = rf_search_results.loc[rf_search_results["validation_f1"].idxmax()]
best_rf_key = best_rf_row["config"]

random_forest = rf_models[best_rf_key]
rf_threshold = float(best_rf_row["threshold"])
rf_val_f1 = float(best_rf_row["validation_f1"])

rf_val_prob = random_forest.predict_proba(X_val_processed)[:, 1]

print("Selected Random Forest config:", best_rf_key)
print("Selected threshold:", round(rf_threshold, 3))
print("Validation F1:", round(rf_val_f1, 4))

Selected Random Forest config: n_estimators=200, max_depth=10
Selected threshold: 0.68
Validation F1: 0.3716


## 3. Gradient Boosting

`GradientBoostingClassifier` in scikit-learn does not have a `class_weight` parameter, so class imbalance is handled instead with `sample_weight` at fit time, computed with `compute_sample_weight("balanced", y_train)`. This gives the minority class more weight without inventing an option the model does not support.

A small grid is tried on the validation set: `n_estimators` in {100, 200} with `learning_rate` in {0.1, 0.05}, `max_depth=3` fixed. Same selection rule as Random Forest: pick the config with the best validation F1.

In [13]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

train_sample_weight = compute_sample_weight("balanced", y_train)

gb_grid = [
    {"n_estimators": 100, "learning_rate": 0.1},
    {"n_estimators": 200, "learning_rate": 0.05}
]

gb_search_results = []
gb_models = {}

for params in gb_grid:
    gb = GradientBoostingClassifier(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train_processed, y_train, sample_weight=train_sample_weight)

    val_prob = gb.predict_proba(X_val_processed)[:, 1]
    threshold, val_f1 = best_threshold(y_val, val_prob)

    key = f"n_estimators={params['n_estimators']}, learning_rate={params['learning_rate']}"
    gb_models[key] = gb

    gb_search_results.append({
        "config": key,
        "threshold": threshold,
        "validation_f1": val_f1
    })

gb_search_results = pd.DataFrame(gb_search_results)
print(gb_search_results)

                                 config  threshold  validation_f1
0   n_estimators=100, learning_rate=0.1       0.68       0.366520
1  n_estimators=200, learning_rate=0.05       0.70       0.367747


In [14]:
best_gb_row = gb_search_results.loc[gb_search_results["validation_f1"].idxmax()]
best_gb_key = best_gb_row["config"]

gradient_boosting = gb_models[best_gb_key]
gb_threshold = float(best_gb_row["threshold"])
gb_val_f1 = float(best_gb_row["validation_f1"])

gb_val_prob = gradient_boosting.predict_proba(X_val_processed)[:, 1]

print("Selected Gradient Boosting config:", best_gb_key)
print("Selected threshold:", round(gb_threshold, 3))
print("Validation F1:", round(gb_val_f1, 4))

Selected Gradient Boosting config: n_estimators=200, learning_rate=0.05
Selected threshold: 0.7
Validation F1: 0.3677


## Validation summary and locked thresholds

All model-selection and threshold decisions above used validation data only. The table below summarizes what was locked before touching the test set.

In [15]:
locked_settings = pd.DataFrame([
    {"model": "Logistic Regression", "config": "class_weight=balanced", "threshold": log_reg_threshold, "validation_f1": log_reg_val_f1},
    {"model": "Random Forest", "config": best_rf_key, "threshold": rf_threshold, "validation_f1": rf_val_f1},
    {"model": "Gradient Boosting", "config": best_gb_key, "threshold": gb_threshold, "validation_f1": gb_val_f1}
])

print(locked_settings.round(4))

                 model                                config  threshold  \
0  Logistic Regression                 class_weight=balanced       0.68   
1        Random Forest        n_estimators=200, max_depth=10       0.68   
2    Gradient Boosting  n_estimators=200, learning_rate=0.05       0.70   

   validation_f1  
0         0.3678  
1         0.3716  
2         0.3677  


## Final test evaluation

Thresholds and hyperparameters are now locked. Each model's test set is evaluated once.

In [16]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, log_loss

def evaluate_on_test(model, threshold, model_name):
    train_prob = model.predict_proba(X_train_processed)[:, 1]
    val_prob = model.predict_proba(X_val_processed)[:, 1]
    test_prob = model.predict_proba(X_test_processed)[:, 1]

    test_pred = (test_prob >= threshold).astype(int)

    cm = confusion_matrix(y_test, test_pred)
    tn, fp, fn, tp = cm.ravel()

    result = {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, test_prob),
        "pr_auc": average_precision_score(y_test, test_prob),
        "train_log_loss": log_loss(y_train, train_prob),
        "validation_log_loss": log_loss(y_val, val_prob),
        "test_log_loss": log_loss(y_test, test_prob),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }
    return result, cm

log_reg_result, log_reg_cm = evaluate_on_test(log_reg, log_reg_threshold, "Logistic Regression")
rf_result, rf_cm = evaluate_on_test(random_forest, rf_threshold, "Random Forest")
gb_result, gb_cm = evaluate_on_test(gradient_boosting, gb_threshold, "Gradient Boosting")

test_results_df = pd.DataFrame([log_reg_result, rf_result, gb_result])
print(test_results_df.round(4))

                 model  threshold  accuracy  precision  recall      f1  \
0  Logistic Regression       0.68    0.8253     0.2690  0.5421  0.3596   
1        Random Forest       0.68    0.8398     0.2806  0.4921  0.3574   
2    Gradient Boosting       0.70    0.8402     0.2849  0.5071  0.3648   

   roc_auc  pr_auc  train_log_loss  validation_log_loss  test_log_loss  \
0   0.8138  0.2886          0.5305               0.5343         0.5281   
1   0.8114  0.2760          0.4941               0.5103         0.5058   
2   0.8152  0.2858          0.5202               0.5269         0.5202   

   true_negatives  false_positives  false_negatives  true_positives  
0           10284             1766              549             650  
1           10537             1513              609             590  
2           10524             1526              591             608  


## Confusion matrices

In [17]:
print("Logistic Regression")
print(log_reg_cm)

print("\nRandom Forest")
print(rf_cm)

print("\nGradient Boosting")
print(gb_cm)

Logistic Regression
[[10284  1766]
 [  549   650]]

Random Forest
[[10537  1513]
 [  609   590]]

Gradient Boosting
[[10524  1526]
 [  591   608]]


## Log loss

In [18]:
print(test_results_df[["model", "train_log_loss", "validation_log_loss", "test_log_loss"]])

                 model  train_log_loss  validation_log_loss  test_log_loss
0  Logistic Regression        0.530469             0.534268       0.528071
1        Random Forest        0.494119             0.510329       0.505806
2    Gradient Boosting        0.520200             0.526881       0.520240


## Save results

In [19]:
results_to_save = test_results_df[[
    "model", "threshold", "accuracy", "precision", "recall", "f1",
    "roc_auc", "pr_auc", "train_log_loss", "validation_log_loss", "test_log_loss",
    "true_negatives", "false_positives", "false_negatives", "true_positives"
]].copy()

results_to_save["validation_f1"] = [log_reg_val_f1, rf_val_f1, gb_val_f1]
results_to_save["config"] = ["class_weight=balanced", best_rf_key, best_gb_key]

results_to_save.to_csv("model_comparison_D2_results.csv", index=False)

print("Saved: model_comparison_D2_results.csv")
results_to_save

Saved: model_comparison_D2_results.csv


,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,train_log_loss,validation_log_loss,test_log_loss,true_negatives,false_positives,false_negatives,true_positives,validation_f1,config
0,Logistic Regression,0.68,0.825270,0.269040,0.542118,0.359613,0.813801,0.288566,0.530469,0.534268,0.528071,10284,1766,549,650,0.367832,class_weight=balanced
1,Random Forest,0.68,0.839837,0.280552,0.492077,0.357359,0.811374,0.275955,0.494119,0.510329,0.505806,10537,1513,609,590,0.371634,"n_estimators=200, max_depth=10"
2,Gradient Boosting,0.70,0.840214,0.284911,0.507089,0.364836,0.815171,0.285799,0.520200,0.526881,0.520240,10524,1526,591,608,0.367747,"n_estimators=200, learning_rate=0.05"


## Interpretation

| | Logistic Regression | Random Forest | Gradient Boosting |
|---|---|---|---|
| Threshold | 0.680 | 0.680 | 0.700 |
| Accuracy | 0.825 | 0.840 | 0.840 |
| Precision | 0.269 | 0.281 | 0.285 |
| Recall | 0.542 | 0.492 | 0.507 |
| F1 | 0.360 | 0.357 | 0.365 |
| ROC-AUC | 0.814 | 0.811 | 0.815 |
| PR-AUC | 0.289 | 0.276 | 0.286 |

Random Forest and Gradient Boosting both beat Logistic Regression on accuracy, but accuracy is misleading here (see the imbalance note below), so that alone does not pick a winner.

Looking at F1, PR-AUC, precision, recall, and ROC-AUC together: the three models are close, with no single model dominating on every metric. Gradient Boosting has the best F1 (0.365) and the best ROC-AUC (0.815), and its precision (0.285) is also the highest of the three. Logistic Regression has the best recall (0.542) and the best PR-AUC (0.289), meaning it ranks positive cases slightly better overall even though its precision is the lowest. Random Forest has the highest accuracy tied with Gradient Boosting, but the worst recall (0.492) and worst PR-AUC (0.276) of the three.

No model clearly dominates across the evaluation metrics. Gradient Boosting has the highest test F1, ROC-AUC, and precision, while Logistic Regression has the highest recall and PR-AUC. The differences are small, so the model classes perform similarly on the D2 feature set. This suggests that for the D2 feature set, the model class matters much less here than the feature representation did in the earlier C -> D1 -> D2 experiments.

## Precision/recall trade-off and class imbalance

Diabetes is rare in this population: 5,994 positive out of 66,242 (about 9.1%). A model that always predicted "no diabetes" would already reach about 91.0% accuracy while catching zero true cases, which is why accuracy alone is not used to judge these models.

Class imbalance was handled differently depending on what each model supports:
- Logistic Regression and Random Forest both accept `class_weight="balanced"` directly.
- Gradient Boosting has no `class_weight` parameter in scikit-learn, so `sample_weight` from `compute_sample_weight("balanced", y_train)` was passed to `fit` instead, which achieves the same reweighting effect during training.

All three models still needed a validation-selected threshold well above the default 0.5 (0.68-0.70) to get a reasonable precision/recall balance, because the balanced training weights push predicted probabilities up for the whole population, not just the positive class.

The recall/precision trade-off shows up clearly across models: Logistic Regression trades some precision for the highest recall, catching more true diabetes cases (650 true positives) at the cost of more false alarms (1,766 false positives). Random Forest goes the other way, with fewer false positives (1,513) but also fewer true positives (590). Gradient Boosting sits in between. Which of these is "better" depends on whether missed diabetes cases (false negatives) or unnecessary follow-up (false positives) is more costly in practice, which is a decision this notebook does not make.